# 020 — Lógica de primer orden y unificación

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("logic", seed=20)
assert result["kind"] == "logic"
assert result["evidence"]
show(result)


## Solución 1 — Traducciones

a) `∀x ((Estudiante(x) ∧ Estudia(x)) → Aprueba(x))`
b) `∃x (Estudiante(x) ∧ ¬Aprueba(x))`
c) `¬∃x ∀y Conoce(x, y)` (equivalente: `∀x ∃y ¬Conoce(x, y)`)
d) `∃x (Rey(x) ∧ ∀y (Rey(y) → y = x))`

El error clásico en (a) sería usar ∧ con ∀ ("todo objeto es estudiante,
estudia y aprueba") y en (b) usar → con ∃ (verdadera con que exista cualquier
no-estudiante).


## Solución 2 — MGUs

a) `{y/Juan, x/Madre(Juan)}`: primero `y/Juan`, luego `x/Madre(y)` ya
sustituido. ✔

b) **Falla tal cual**: `x` tendría que ser `Juan` (primer argumento) y `Elisa`
(segundo) a la vez. La causa es no haber **estandarizado aparte**: renombrando
la segunda a `Conoce(z, Elisa)`, la MGU es `{z/Juan, x/Elisa}`.

c) **Falla por occurs check**: `y/x` y luego `x` debe unificar con `f(x)` —
`x` ocurre dentro del término. No hay unificador finito.

d) `y/f(a)` y luego `g(x)` debe unificar con `y = f(a)`: **falla** porque los
símbolos de función `g` y `f` difieren.


In [ ]:
mgu_a = {"y": "Juan", "x": "Madre(Juan)"}
mgu_b = {"z": "Juan", "x": "Elisa"}  # tras estandarizar aparte
mgu_c = None  # occurs check
mgu_d = None  # funciones g/f incompatibles
print("MGUs registradas ✔")


## Solución 3 — Modus ponens generalizado

a) Con `{x/pipeline1}` ambas premisas unifican con hechos de la KB →
se concluye `Despliega(pipeline1)`.

b) Para `{x/pipeline2}` solo `Rapido(pipeline2)` está en la KB; falta
`Seguro(pipeline2)` y la regla exige **ambas** premisas. La inferencia correcta
no rellena huecos.

c) Añadir el hecho `Seguro(pipeline2)`.


## Solución 4 — Una regla con variables

```text
∀p (TieneDatos(p) ∧ TieneObjetivo(p) → PuedeExperimentar(p))
∀p (PuedeExperimentar(p) → RequiereBaseline(p))
∀p (RequiereBaseline(p) → RequiereEvaluacion(p))
```

Cada regla de primer orden sustituye a **una copia por proyecto**: con 100
proyectos, las 3 reglas con variables reemplazan 300 reglas proposicionales.
Ese es el poder expresivo de FOL; el costo nuevo es que cada disparo exige
unificar premisas contra hechos (el matching deja de ser una comprobación de
pertenencia a un conjunto y pasa a ser búsqueda de sustituciones).


In [ ]:
result = run_lab("logic", seed=20)
assert len(result["result"]["rules_fired"]) == 3
print("3 reglas proposicionales = 1 pasada de la versión cuantificada para este 'proyecto' ✔")


## Reflexión

1. ¿Por qué ∀ se combina con → y ∃ con ∧? Escribe las dos combinaciones 'incorrectas' para 'todos los gatos duermen' y explica qué afirman realmente.
2. Sin occurs check, unificar P(x, x) con P(y, f(y)) 'tendría éxito' con un término infinito. ¿Por qué Prolog lo omite por defecto y qué riesgo acepta a cambio?
3. El laboratorio encadena reglas proposicionales (sin variables). ¿Qué gana exactamente una base de reglas al pasar a primer orden y qué nuevo costo computacional aparece con la unificación?
